In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"), temperature=0.4)

c:\Users\harih\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sqlite3
from datetime import datetime, timezone

conn = sqlite3.connect("notes.db", check_same_thread=False)
conn.row_factory = sqlite3.Row

conn.execute("""
CREATE TABLE IF NOT EXISTS notes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    content TEXT NOT NULL,
    tags TEXT DEFAULT '',
    created_at TEXT NOT NULL
)
""")
conn.commit()

print("notes table ready")

notes table ready


In [3]:
from langchain_core.tools import tool

def now():
    return datetime.now(timezone.utc).isoformat()

@tool
def add_note(title: str, content: str, tags: str = "") -> str:
    """Save a new note with a title, content, and optional comma-separated tags."""
    cur = conn.execute(
        "INSERT INTO notes (title, content, tags, created_at) VALUES (?, ?, ?, ?)",
        (title, content, tags, now())
    )
    conn.commit()
    return f"Note saved with id {cur.lastrowid}: '{title}'."

@tool
def list_notes(limit: int = 20) -> str:
    """List the most recent notes."""
    rows = conn.execute(
        "SELECT id, title, tags, created_at FROM notes ORDER BY created_at DESC LIMIT ?",
        (limit,)
    ).fetchall()
    if not rows:
        return "No notes found."
    return "\n".join(f"[{r['id']}] {r['title']} (tags: {r['tags']})" for r in rows)

@tool
def search_notes(query: str) -> str:
    """Search notes by keyword in title, content, or tags."""
    like = f"%{query}%"
    rows = conn.execute(
        "SELECT id, title, content, tags FROM notes WHERE title LIKE ? OR content LIKE ? OR tags LIKE ?",
        (like, like, like)
    ).fetchall()
    if not rows:
        return f"No notes matched '{query}'."
    return "\n\n".join(f"[{r['id']}] {r['title']}\n{r['content']}" for r in rows)

@tool
def update_note(note_id: int, title: str = None, content: str = None, tags: str = None) -> str:
    """Update an existing note's title, content, and/or tags by id."""
    row = conn.execute("SELECT * FROM notes WHERE id = ?", (note_id,)).fetchone()
    if not row:
        return f"No note found with id {note_id}."
    conn.execute(
        "UPDATE notes SET title = ?, content = ?, tags = ? WHERE id = ?",
        (title or row["title"], content or row["content"], tags or row["tags"], note_id)
    )
    conn.commit()
    return f"Note {note_id} updated."

@tool
def delete_note(note_id: int) -> str:
    """Delete a note by id."""
    conn.execute("DELETE FROM notes WHERE id = ?", (note_id,))
    conn.commit()
    return f"Note {note_id} deleted."

notes_toolkit = [add_note, list_notes, search_notes, update_note, delete_note]

In [4]:
print(add_note.invoke({"title": "Client ABC kickoff", "content": "Discussed scope and timeline for Q3 project.", "tags": "client,meeting"}))

Note saved with id 1: 'Client ABC kickoff'.


In [5]:
from langchain.agents import create_agent
from datetime import date

today_str = date.today().strftime("%B %d, %Y")

system_prompt = f"""You are a notes management assistant. Today's date is {today_str}.

You have tools to add, list, search, update, and delete notes.

Rules:
- Use the most specific tool for the request (e.g. search_notes for finding something, not list_notes).
- Call each tool AT MOST ONCE per user request unless the result clearly requires a follow-up action (e.g. search then delete).
- After using tools, give a clear, concise confirmation or summary. Do not repeat tool calls "to be safe"."""

agent = create_agent(llm, tools=notes_toolkit, system_prompt=system_prompt)

In [7]:
example_query = "Save a note about tomorrow's client presentation. Title it 'Client Presentation Prep' and tag it as work,presentation."

events = agent.stream(
    {"messages": [("user", example_query)]},
    config={"recursion_limit": 8},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Save a note about tomorrow's client presentation. Title it 'Client Presentation Prep' and tag it as work,presentation.
================================== Ai Message ==================================
Tool Calls:
  add_note (fc_dbeed212-0fff-4eb3-9661-57e5f5749da6)
 Call ID: fc_dbeed212-0fff-4eb3-9661-57e5f5749da6
  Args:
    content: Tomorrow's client presentation.
    tags: work,presentation
    title: Client Presentation Prep
================================= Tool Message =================================
Name: add_note

Note saved with id 3: 'Client Presentation Prep'.
================================== Ai Message ==================================

Your note has been saved:

- **ID:** 3  
- **Title:** Client Presentation Prep  
- **Tags:** work, presentation  
- **Content:** Tomorrow's client presentation.


In [8]:
example_query = "Show all notes related to clients."

events = agent.stream(
    {"messages": [("user", example_query)]},
    config={"recursion_limit": 8},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Show all notes related to clients.
================================== Ai Message ==================================
Tool Calls:
  search_notes (fc_36a3d5ce-b588-4f4e-91fa-ed4065d1da6e)
 Call ID: fc_36a3d5ce-b588-4f4e-91fa-ed4065d1da6e
  Args:
    query: clients
================================= Tool Message =================================
Name: search_notes

No notes matched 'clients'.
================================== Ai Message ==================================

I searched your notes for the keyword **“clients”**, but there are no matching notes at the moment. Let me know if you’d like to create a new note or adjust the search criteria.


In [9]:
@tool
def search_notes(query: str) -> str:
    """Search notes by keyword in title, content, or tags. Matches any significant word in the query."""
    words = [w.strip() for w in query.lower().split() if len(w.strip()) > 2]
    if not words:
        return "Please provide a more specific search term."

    conditions = " OR ".join(["(LOWER(title) LIKE ? OR LOWER(content) LIKE ? OR LOWER(tags) LIKE ?)"] * len(words))
    params = []
    for w in words:
        like = f"%{w}%"
        params.extend([like, like, like])

    rows = conn.execute(f"SELECT id, title, content, tags FROM notes WHERE {conditions}", params).fetchall()
    if not rows:
        return f"No notes matched '{query}'."
    return "\n\n".join(f"[{r['id']}] {r['title']}\n{r['content']}" for r in rows)

In [10]:
notes_toolkit = [add_note, list_notes, search_notes, update_note, delete_note]

In [11]:
agent = create_agent(llm, tools=notes_toolkit, system_prompt=system_prompt)

In [12]:
example_query = "Show all notes related to clients."

events = agent.stream(
    {"messages": [("user", example_query)]},
    config={"recursion_limit": 8},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Show all notes related to clients.
================================== Ai Message ==================================
Tool Calls:
  search_notes (fc_cb69295f-45e8-40d7-a386-35d0df1cb730)
 Call ID: fc_cb69295f-45e8-40d7-a386-35d0df1cb730
  Args:
    query: clients
================================= Tool Message =================================
Name: search_notes

No notes matched 'clients'.
================================== Ai Message ==================================

I searched your notes for the keyword **“clients,”** but there are no matching entries at the moment. Let me know if you’d like to create a new note or try a different search term!


In [13]:
@tool
def search_notes(query: str) -> str:
    """Search notes by keyword in title, content, or tags. Matches singular/plural word forms."""
    words = [w.strip().lower() for w in query.split() if len(w.strip()) > 2]
    if not words:
        return "Please provide a more specific search term."

    # naive singular/plural handling: also try the word with trailing 's' stripped or added
    variants = set()
    for w in words:
        variants.add(w)
        if w.endswith("s"):
            variants.add(w[:-1])
        else:
            variants.add(w + "s")

    conditions = " OR ".join(["(LOWER(title) LIKE ? OR LOWER(content) LIKE ? OR LOWER(tags) LIKE ?)"] * len(variants))
    params = []
    for v in variants:
        like = f"%{v}%"
        params.extend([like, like, like])

    rows = conn.execute(f"SELECT id, title, content, tags FROM notes WHERE {conditions}", params).fetchall()
    if not rows:
        return f"No notes matched '{query}'."
    return "\n\n".join(f"[{r['id']}] {r['title']}\n{r['content']}" for r in rows)

In [14]:
notes_toolkit = [add_note, list_notes, search_notes, update_note, delete_note]
agent = create_agent(llm, tools=notes_toolkit, system_prompt=system_prompt)

In [15]:
example_query = "Show all notes related to clients."

events = agent.stream(
    {"messages": [("user", example_query)]},
    config={"recursion_limit": 8},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Show all notes related to clients.
================================== Ai Message ==================================
Tool Calls:
  search_notes (fc_54dd5caa-4fef-4325-96ee-4fb359f6104f)
 Call ID: fc_54dd5caa-4fef-4325-96ee-4fb359f6104f
  Args:
    query: clients
================================= Tool Message =================================
Name: search_notes

[1] Client ABC kickoff
Discussed scope and timeline for Q3 project.

[2] Client Presentation Prep
Prepare slides and talking points for tomorrow's client presentation.

[3] Client Presentation Prep
Tomorrow's client presentation.
================================== Ai Message ==================================

Here are the notes that mention **clients**:

1. **Client ABC kickoff**  
   *Content:* Discussed scope and timeline for Q3 project.

2. **Client Presentation Prep**  
   *Content:* Prepare slides and talking points for tomorrow's client prese